<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

In [1]:
# Parameters
execute_all = "true"


# NBA Player Performance EDA: Rigorous Statistical Analysis

**Dataset:** combined_player_stats (136,965 rows × 428 columns)

**Purpose:** Characterize the dataset structure, quality, and statistical properties to inform downstream modeling decisions.

**Scope:** This notebook focuses on **exploratory data analysis only**:
- Data quality assessment (missing values, outliers, anomalies)
- Distributional analysis with confidence intervals
- Bivariate relationships (correlations, not predictions)
- Stratified summaries by player tier and position
- Temporal patterns and trends

**Out of Scope:** Model fitting, cross-validation, prediction evaluation, betting strategy.

---

## Table of Contents

1. [Configuration & Data Loading](#1-configuration)
2. [Data Quality Assessment](#2-data-quality)
3. [Univariate Distributions](#3-distributions)
4. [Bivariate Relationships](#4-bivariate)
5. [Game-Level Analysis](#5-games)
6. [Stratified Analysis](#6-stratified)
7. [Temporal Patterns](#7-temporal)
8. [Publication-Quality Visualizations](#8-visualizations)
9. [Summary & Recommendations](#9-summary)

---

## 1. Configuration & Data Loading <a id="1-configuration"></a>

In [2]:
import os
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sqlalchemy import create_engine

# Statistical corrections
from statsmodels.stats.multitest import fdrcorrection
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')  # Accessibility

np.random.seed(42)  # Reproducibility for bootstrap

In [3]:
@dataclass
class Config:
    """Analysis configuration for multi-season EDA."""
    # Multi-season support
    seasons: list = None  # Will be set after init
    
    min_games_played: int = 10

    # Statistical thresholds
    alpha: float = 0.05
    ci_level: float = 0.95
    n_bootstrap: int = 1000

    # Player tier thresholds (PPG)
    star_threshold: float = 20.0
    rotation_threshold: float = 8.0

    # Outlier detection
    iqr_multiplier: float = 1.5

    # Output
    fig_dir: Path = Path('figures')
    fig_dpi: int = 150
    
    def __post_init__(self):
        if self.seasons is None:
            self.seasons = ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

CONFIG = Config()
CONFIG.fig_dir.mkdir(exist_ok=True)
print(f"Analyzing seasons: {CONFIG.seasons}")
print(f"Significance level: α = {CONFIG.alpha}")

Analyzing seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Significance level: α = 0.05


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
# Database connection
load_dotenv(Path('..', '..', '..', '.env'))
DATABASE_URL = os.getenv('DATABASE_URL')
engine = create_engine(DATABASE_URL)

def query(sql: str) -> pd.DataFrame:
    """Execute SQL and return DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql_query(sql, conn)

# Verify connection
overview = query("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT game_id) as games,
        COUNT(DISTINCT player_id) as players,
        COUNT(DISTINCT team_abbr) as teams,
        MIN(game_date) as first_game,
        MAX(game_date) as last_game
    FROM combined_player_stats
""")
print("Full Dataset Overview:")
print(f"  Rows: {overview['total_rows'].iloc[0]:,}")
print(f"  Games: {overview['games'].iloc[0]:,}")
print(f"  Players: {overview['players'].iloc[0]:,}")
print(f"  Teams: {overview['teams'].iloc[0]}")
print(f"  Date range: {overview['first_game'].iloc[0]} to {overview['last_game'].iloc[0]}")

ArgumentError: Expected string or URL object, got None

In [ ]:
# Load ALL seasons data (2021-22 through 2025-26)
df = query("""
    SELECT * FROM combined_player_stats
    ORDER BY game_date, game_id, player_id
""")

# Parse dates
df['game_date'] = pd.to_datetime(df['game_date'])

# Parse minutes to float
def parse_minutes(min_str) -> float:
    if pd.isna(min_str) or str(min_str).strip() == '':
        return 0.0
    s = str(min_str).strip()
    if ':' in s:
        parts = s.split(':')
        try:
            return int(parts[0]) + int(parts[1]) / 60
        except (ValueError, IndexError):
            return 0.0
    try:
        return float(s)
    except ValueError:
        return 0.0

df['minutes'] = df['min'].apply(parse_minutes)

# Season-level summary
season_summary = df.groupby('season').agg({
    'game_id': 'nunique',
    'player_id': 'nunique',
    'game_date': ['min', 'max']
}).reset_index()
season_summary.columns = ['Season', 'Games', 'Players', 'First Game', 'Last Game']

print(f"\n=== FULL DATASET ===")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Seasons: {df['season'].nunique()}")
print(f"Date range: {df['game_date'].min().date()} to {df['game_date'].max().date()}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\n=== BY SEASON ===")
print(season_summary.to_string(index=False))

In [ ]:
# Column categorization
cols = df.columns.tolist()

col_groups = {
    'Identifiers': ['game_id', 'player_id', 'player_name', 'team_id', 'team_abbr', 'game_date', 'season'],
    'Boxscore': ['pts', 'reb', 'ast', 'stl', 'blk', 'tov', 'pf', 'fg_pct', 'fg3_pct', 'ft_pct', 'min'],
    'Advanced': ['ts_pct', 'usg_pct', 'off_rating', 'def_rating', 'net_rating', 'pie'],
    'Pregame (player)': [c for c in cols if c.startswith(('season_avg', 'last5_avg', 'last10_'))],
    'Pregame (team)': [c for c in cols if c.startswith('team_pre_')],
    'Quarter': [c for c in cols if c.startswith(('q1_', 'q2_', 'q3_', 'q4_', 'ot'))],
}

print("Column Groups:")
for group, group_cols in col_groups.items():
    count = len([c for c in group_cols if c in cols])
    print(f"  {group}: {count}")

---

## 2. Data Quality Assessment <a id="2-data-quality"></a>

In [ ]:
# Missing data by column group
def missing_by_group(df: pd.DataFrame, group_cols: List[str]) -> float:
    valid = [c for c in group_cols if c in df.columns]
    if not valid:
        return np.nan
    return df[valid].isna().mean().mean() * 100

missing_summary = {}
for group, group_cols in col_groups.items():
    missing_summary[group] = missing_by_group(df, group_cols)

print("Missing Data by Category:")
for group, pct in missing_summary.items():
    if not np.isnan(pct):
        print(f"  {group}: {pct:.1f}%")

In [ ]:
# Missing data mechanism analysis
# MCAR: Missing Completely At Random
# MAR: Missing At Random (conditional on observed data)
# MNAR: Missing Not At Random (depends on unobserved value)

def analyze_missing(df: pd.DataFrame, col: str, predictor: str) -> dict:
    """Test if missingness in col is related to predictor.
    
    Uses point-biserial correlation between missingness indicator and predictor.
    """
    if col not in df.columns or predictor not in df.columns:
        return {'column': col, 'predictor': predictor, 'error': 'column_not_found'}

    missing = df[col].isna().astype(int)
    valid_mask = df[predictor].notna()

    if valid_mask.sum() < 30:
        return {'column': col, 'predictor': predictor, 'error': 'insufficient_data'}
    
    # Check for variance in both variables (required for correlation)
    missing_subset = missing[valid_mask]
    predictor_subset = df.loc[valid_mask, predictor]
    
    if missing_subset.std() == 0:
        # All missing or all present - can't compute correlation
        missing_rate = missing_subset.mean()
        return {
            'column': col, 
            'predictor': predictor, 
            'note': f'No variance in missingness (rate={missing_rate:.1%})',
            'mechanism': 'Cannot test - no missingness variance'
        }
    
    if predictor_subset.std() == 0:
        return {'column': col, 'predictor': predictor, 'error': 'no_variance_in_predictor'}

    r, p = pearsonr(missing_subset, predictor_subset)

    # Classify mechanism
    if p >= CONFIG.alpha:
        mechanism = 'MCAR (unrelated to predictor)'
    elif abs(r) < 0.3:
        mechanism = 'MAR (weakly related)'
    else:
        mechanism = 'MAR/MNAR (strongly related)'

    return {
        'column': col, 
        'predictor': predictor, 
        'r': r, 
        'p': p, 
        'mechanism': mechanism,
        'n': int(valid_mask.sum()),
    }

# Test key relationships
tests = [
    ('season_avg_pts', 'season_gp'),  # First games have no avg
    ('speed', 'minutes'),              # Tracking data availability
    ('ts_pct', 'minutes'),             # Low-minute players
]

print("Missing Data Mechanism Analysis:")
print("-" * 60)
for col, pred in tests:
    result = analyze_missing(df, col, pred)
    print(f"\n{result['column']} ~ {result['predictor']}:")
    if 'error' in result:
        print(f"  ⚠ Error: {result['error']}")
    elif 'note' in result:
        print(f"  ℹ {result['note']}")
        print(f"  → {result['mechanism']}")
    else:
        print(f"  n = {result['n']:,}")
        print(f"  r = {result['r']:.3f}, p = {result['p']:.4f}")
        print(f"  → {result['mechanism']}")

In [ ]:
# Outlier detection (descriptive, not removal)
played_df = df[df['played'] == 1].copy()

def outlier_summary(series: pd.Series, name: str) -> dict:
    """IQR-based outlier detection."""
    valid = series.dropna()
    if len(valid) < 30:
        return None

    Q1, Q3 = valid.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - CONFIG.iqr_multiplier * IQR
    upper = Q3 + CONFIG.iqr_multiplier * IQR

    outliers = (valid < lower) | (valid > upper)

    return {
        'variable': name,
        'n': len(valid),
        'outliers': outliers.sum(),
        'pct': outliers.mean() * 100,
        'lower_bound': lower,
        'upper_bound': upper,
        'min': valid.min(),
        'max': valid.max(),
    }

outlier_results = []
for col in ['pts', 'reb', 'ast', 'minutes', 'ts_pct', 'usg_pct']:
    if col in played_df.columns:
        result = outlier_summary(played_df[col], col.upper())
        if result:
            outlier_results.append(result)

outlier_df = pd.DataFrame(outlier_results)
outlier_df['pct'] = outlier_df['pct'].apply(lambda x: f"{x:.1f}%")
outlier_df['range'] = outlier_df.apply(
    lambda r: f"[{r['min']:.1f}, {r['max']:.1f}]", axis=1
)
print("Outlier Summary (IQR method):")
print(outlier_df[['variable', 'n', 'outliers', 'pct', 'range']].to_string(index=False))

In [ ]:
# Data integrity checks
print("Data Integrity Checks:")
print("-" * 40)

# Duplicates
dup_count = df.duplicated(subset=['game_id', 'player_id']).sum()
print(f"Duplicate (game_id, player_id): {dup_count}")

# Value ranges
range_checks = [
    ('fg_pct', 0, 1),
    ('ts_pct', 0, 1.5),
    ('usg_pct', 0, 1),
    ('pts', 0, 100),
]

for col, min_v, max_v in range_checks:
    if col in played_df.columns:
        valid = played_df[col].dropna()
        below = (valid < min_v).sum()
        above = (valid > max_v).sum()
        status = "OK" if below == 0 and above == 0 else f"{below} below, {above} above"
        print(f"{col} in [{min_v}, {max_v}]: {status}")

# Quarter sum consistency
q_cols = ['q1_pts', 'q2_pts', 'q3_pts', 'q4_pts']
if all(c in df.columns for c in q_cols):
    df['q_sum'] = df[q_cols].fillna(0).sum(axis=1)
    # Add OT if present
    for ot in ['ot1_pts', 'ot2_pts', 'ot3_pts']:
        if ot in df.columns:
            df['q_sum'] += df[ot].fillna(0)

    match_rate = (df['pts'] == df['q_sum']).mean() * 100
    print(f"Quarter pts sum matches total: {match_rate:.1f}%")

---

## 3. Univariate Distributions <a id="3-distributions"></a>

In [ ]:
def summarize_distribution(series: pd.Series, n_bootstrap: int = 1000) -> dict:
    """
    Comprehensive distribution summary with bootstrap CI for mean.
    """
    data = series.dropna().values
    n = len(data)

    if n < 30:
        return {'n': n, 'error': 'insufficient_data'}

    # Point estimates
    mean = np.mean(data)
    median = np.median(data)
    std = np.std(data, ddof=1)
    skew = stats.skew(data)
    kurtosis = stats.kurtosis(data)

    # Bootstrap CI for mean
    boot_means = [np.mean(np.random.choice(data, n, replace=True)) for _ in range(n_bootstrap)]
    ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])

    # Normality test
    if n <= 5000:
        _, norm_p = stats.normaltest(data) if n > 50 else stats.shapiro(data)
    else:
        _, norm_p = stats.normaltest(np.random.choice(data, 5000, replace=False))

    return {
        'n': n,
        'mean': mean,
        'ci_95': (ci_lower, ci_upper),
        'median': median,
        'std': std,
        'cv': std / mean if mean != 0 else np.nan,  # Coefficient of variation
        'skewness': skew,
        'kurtosis': kurtosis,
        'normal_p': norm_p,
        'is_normal': norm_p > CONFIG.alpha,
        'min': np.min(data),
        'max': np.max(data),
        'p25': np.percentile(data, 25),
        'p75': np.percentile(data, 75),
    }


def clustered_bootstrap_ci(
    df: pd.DataFrame, 
    col: str, 
    cluster_col: str = 'player_id', 
    n_bootstrap: int = 1000,
    ci_level: float = 0.95
) -> Tuple[float, float, float]:
    """
    Bootstrap CI by resampling clusters (e.g., players), not individual observations.
    
    This accounts for within-cluster correlation (e.g., repeated measures on same player).
    
    Returns:
        Tuple of (mean, ci_lower, ci_upper)
    """
    data = df[[col, cluster_col]].dropna()
    clusters = data[cluster_col].unique()
    n_clusters = len(clusters)
    
    if n_clusters < 30:
        return (data[col].mean(), np.nan, np.nan)
    
    boot_means = []
    for _ in range(n_bootstrap):
        # Resample clusters with replacement
        sampled_clusters = np.random.choice(clusters, n_clusters, replace=True)
        # Get all observations from sampled clusters
        boot_sample = data[data[cluster_col].isin(sampled_clusters)][col]
        boot_means.append(boot_sample.mean())
    
    alpha = 1 - ci_level
    ci_lower, ci_upper = np.percentile(boot_means, [100*alpha/2, 100*(1-alpha/2)])
    
    return (data[col].mean(), ci_lower, ci_upper)

In [ ]:
# Target variable distributions
targets = ['pts', 'reb', 'ast']

target_stats = []
for col in targets:
    result = summarize_distribution(played_df[col], CONFIG.n_bootstrap)
    result['variable'] = col.upper()
    target_stats.append(result)

# Format display
display_df = pd.DataFrame([{
    'Variable': r['variable'],
    'N': f"{r['n']:,}",
    'Mean': f"{r['mean']:.2f}",
    '95% CI': f"[{r['ci_95'][0]:.2f}, {r['ci_95'][1]:.2f}]",
    'Median': f"{r['median']:.2f}",
    'SD': f"{r['std']:.2f}",
    'CV': f"{r['cv']:.2f}",
    'Skew': f"{r['skewness']:.2f}",
    'Normal': 'Yes' if r['is_normal'] else 'No',
} for r in target_stats])

print("Target Variable Distributions:")
print(display_df.to_string(index=False))
print("\nCV = Coefficient of Variation (SD/Mean)")
print("Higher CV indicates more relative variability.")
print("\n⚠ Note on normality tests: With large samples (n > 10,000), normality tests")
print("  will almost always reject H₀ even for approximately normal data. This does NOT")
print("  mean parametric methods are inappropriate - the Central Limit Theorem ensures")
print("  inference on means remains valid. The skewness values are more informative.")

In [ ]:
# Distribution visualizations
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, col in enumerate(targets):
    data = played_df[col].dropna()
    result = target_stats[i]

    # Histogram
    ax = axes[0, i]
    ax.hist(data, bins=50, density=True, alpha=0.7, edgecolor='white')

    # Mean with CI
    ax.axvline(result['mean'], color='red', linestyle='-', linewidth=2, label='Mean')
    ax.axvline(result['median'], color='orange', linestyle='--', linewidth=2, label='Median')
    ax.axvspan(result['ci_95'][0], result['ci_95'][1], alpha=0.2, color='red')

    ax.set_xlabel(col.upper())
    ax.set_ylabel('Density')
    ax.set_title(f"{col.upper()} (Skew={result['skewness']:.2f})")
    ax.legend(fontsize=8)

    # Q-Q plot
    ax_qq = axes[1, i]
    stats.probplot(data, dist="norm", plot=ax_qq)
    ax_qq.set_title(f"{col.upper()} Q-Q Plot")

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'target_distributions.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

print("\nKey Observation: All targets show positive skew (right tail).")
print("This is expected - most players score modestly, few score very high.")

In [ ]:
# Key feature distributions
features = ['minutes', 'ts_pct', 'usg_pct', 'season_avg_pts', 'last5_avg_pts']

feature_stats = []
for col in features:
    if col in played_df.columns:
        result = summarize_distribution(played_df[col], CONFIG.n_bootstrap)
        result['variable'] = col
        feature_stats.append(result)

feat_display = pd.DataFrame([{
    'Feature': r['variable'],
    'N': f"{r['n']:,}",
    'Mean (95% CI)': f"{r['mean']:.2f} [{r['ci_95'][0]:.2f}, {r['ci_95'][1]:.2f}]",
    'Median': f"{r['median']:.2f}",
    'Range': f"[{r['min']:.1f}, {r['max']:.1f}]",
} for r in feature_stats if 'error' not in r])

print("Key Feature Distributions:")
print(feat_display.to_string(index=False))

---

## 4. Bivariate Relationships <a id="4-bivariate"></a>

**Note:** These are descriptive correlations characterizing relationships in the data. They are not predictions.

In [ ]:
def correlation_with_ci(
    x: pd.Series,
    y: pd.Series,
    method: str = 'pearson',
    n_bootstrap: int = 1000
) -> dict:
    """
    Calculate correlation with bootstrap CI.
    """
    valid = pd.DataFrame({'x': x, 'y': y}).dropna()
    x_vals, y_vals = valid['x'].values, valid['y'].values
    n = len(x_vals)

    if n < 30:
        return {'n': n, 'error': 'insufficient_data'}

    # Point estimate
    if method == 'pearson':
        r, p = pearsonr(x_vals, y_vals)
    else:
        r, p = spearmanr(x_vals, y_vals)

    # Bootstrap CI
    boot_r = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        if method == 'pearson':
            boot_r.append(pearsonr(x_vals[idx], y_vals[idx])[0])
        else:
            boot_r.append(spearmanr(x_vals[idx], y_vals[idx])[0])

    ci_lower, ci_upper = np.percentile(boot_r, [2.5, 97.5])

    return {
        'n': n,
        'r': r,
        'r_squared': r**2,
        'ci_95': (ci_lower, ci_upper),
        'p_value': p,
        'significant': p < CONFIG.alpha,
    }

In [ ]:
# Key bivariate relationships
pairs = [
    ('pts', 'season_avg_pts', 'Points vs Season Avg'),
    ('pts', 'last5_avg_pts', 'Points vs Last 5 Avg'),
    ('pts', 'minutes', 'Points vs Minutes'),
    ('pts', 'usg_pct', 'Points vs Usage Rate'),
    ('reb', 'season_avg_reb', 'Rebounds vs Season Avg'),
    ('ast', 'season_avg_ast', 'Assists vs Season Avg'),
]

corr_results = []
for x_col, y_col, label in pairs:
    if x_col in played_df.columns and y_col in played_df.columns:
        result = correlation_with_ci(played_df[x_col], played_df[y_col], n_bootstrap=CONFIG.n_bootstrap)
        if 'error' not in result:
            result['comparison'] = label
            corr_results.append(result)

# Apply FDR correction for multiple comparisons
p_values = [r['p_value'] for r in corr_results]
reject, p_adjusted = fdrcorrection(p_values, alpha=CONFIG.alpha)

for i, r in enumerate(corr_results):
    r['p_adjusted'] = p_adjusted[i]
    r['significant_fdr'] = reject[i]

corr_display = pd.DataFrame([{
    'Relationship': r['comparison'],
    'N': f"{r['n']:,}",
    'r': f"{r['r']:.3f}",
    'R²': f"{r['r_squared']:.3f}",
    '95% CI': f"[{r['ci_95'][0]:.3f}, {r['ci_95'][1]:.3f}]",
    'p (raw)': f"{r['p_value']:.2e}",
    'p (FDR)': f"{r['p_adjusted']:.2e}",
    'Sig': '✓' if r['significant_fdr'] else '',
} for r in corr_results])

print("Key Bivariate Correlations (with 95% CI and FDR correction):")
print(corr_display.to_string(index=False))
print("\nNote: p (FDR) = Benjamini-Hochberg adjusted p-values for multiple testing.")
print("      Sig = significant after FDR correction at α = 0.05")
print("      These describe linear association, not causal relationships.")

In [ ]:
# Scatter plots for key relationships
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

plot_pairs = [
    ('season_avg_pts', 'pts', 'Season Avg → Game PTS'),
    ('minutes', 'pts', 'Minutes → Game PTS'),
    ('season_avg_reb', 'reb', 'Season Avg → Game REB'),
]

for i, (x_col, y_col, title) in enumerate(plot_pairs):
    if x_col in played_df.columns and y_col in played_df.columns:
        valid = played_df[[x_col, y_col]].dropna()
        r = valid[x_col].corr(valid[y_col])

        axes[i].scatter(valid[x_col], valid[y_col], alpha=0.1, s=5)
        axes[i].set_xlabel(x_col)
        axes[i].set_ylabel(y_col)
        axes[i].set_title(f"{title}\n(r = {r:.2f})")

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'scatter_relationships.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix for key features
corr_cols = ['pts', 'reb', 'ast', 'minutes', 'ts_pct', 'usg_pct', 'plus_minus']
corr_cols = [c for c in corr_cols if c in played_df.columns]

corr_matrix = played_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'correlation_matrix.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

---

## 5. Game-Level Analysis <a id="5-games"></a>

In [ ]:
# Construct game-level dataset
game_teams = df.groupby(['game_id', 'team_abbr']).agg({
    'team_pts': 'first',
    'team_pre_season_w_pct': 'first',
    'team_pre_season_ppg': 'first',
    'team_pre_is_home': 'first',
    'team_pre_days_rest': 'first',
    'game_date': 'first',
}).reset_index()

# Build home/away game records
games = []
for game_id, group in game_teams.groupby('game_id'):
    if len(group) != 2:
        continue

    home = group[group['team_pre_is_home'] == 1]
    away = group[group['team_pre_is_home'] == 0]

    if len(home) != 1 or len(away) != 1:
        continue

    home, away = home.iloc[0], away.iloc[0]

    games.append({
        'game_id': game_id,
        'game_date': home['game_date'],
        'home_team': home['team_abbr'],
        'away_team': away['team_abbr'],
        'home_pts': home['team_pts'],
        'away_pts': away['team_pts'],
        'home_win': 1 if home['team_pts'] > away['team_pts'] else 0,
        'point_diff': home['team_pts'] - away['team_pts'],
        'total_pts': home['team_pts'] + away['team_pts'],
        'home_pre_w_pct': home['team_pre_season_w_pct'],
        'away_pre_w_pct': away['team_pre_season_w_pct'],
    })

games_df = pd.DataFrame(games).dropna()
print(f"Constructed {len(games_df):,} games with complete data")

In [ ]:
# Home court advantage (descriptive)
home_wins = games_df['home_win'].sum()
total_games = len(games_df)
home_win_rate = home_wins / total_games

# Wilson score interval for proportion (preferred over Clopper-Pearson for moderate n)
# Using statsmodels for correct implementation
ci_lower, ci_upper = proportion_confint(
    home_wins, 
    total_games, 
    alpha=1 - CONFIG.ci_level, 
    method='wilson'
)

# Point differential with CI
mean_diff = games_df['point_diff'].mean()
se_diff = games_df['point_diff'].std() / np.sqrt(total_games)

# Cohen's h effect size for proportion vs 0.5
cohens_h = 2 * (np.arcsin(np.sqrt(home_win_rate)) - np.arcsin(np.sqrt(0.5)))

print("Home Court Advantage Summary")
print("=" * 40)
print(f"Games analyzed: {total_games:,}")
print(f"Home wins: {home_wins:,}")
print(f"Home win rate: {home_win_rate*100:.1f}%")
print(f"95% Wilson CI: [{ci_lower*100:.1f}%, {ci_upper*100:.1f}%]")
print(f"\nMean point differential: {mean_diff:+.2f}")
print(f"95% CI: [{mean_diff - 1.96*se_diff:+.2f}, {mean_diff + 1.96*se_diff:+.2f}]")
print(f"\nCohen's h effect size: {cohens_h:.3f}")
print(f"Interpretation: {'Small' if abs(cohens_h) < 0.5 else 'Medium'} effect")
print("\nNote: Wilson CI is preferred over Clopper-Pearson for better coverage properties.")

In [ ]:
# Game-level distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Point differential
axes[0].hist(games_df['point_diff'], bins=40, edgecolor='white', alpha=0.7)
axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
axes[0].axvline(mean_diff, color='red', linestyle='-', linewidth=2, label=f'Mean: {mean_diff:+.1f}')
axes[0].set_xlabel('Point Differential (Home - Away)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Home Point Differential')
axes[0].legend()

# Total points
total_mean = games_df['total_pts'].mean()
axes[1].hist(games_df['total_pts'], bins=40, edgecolor='white', alpha=0.7)
axes[1].axvline(total_mean, color='red', linestyle='-', linewidth=2, label=f'Mean: {total_mean:.0f}')
axes[1].set_xlabel('Total Points')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Game Total Points')
axes[1].legend()

# Home vs Away points
axes[2].scatter(games_df['away_pts'], games_df['home_pts'], alpha=0.3, s=20)
max_pts = max(games_df['home_pts'].max(), games_df['away_pts'].max())
axes[2].plot([0, max_pts], [0, max_pts], 'r--', label='Equal')
axes[2].set_xlabel('Away Points')
axes[2].set_ylabel('Home Points')
axes[2].set_title('Home vs Away Scoring')
axes[2].legend()

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'game_distributions.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

---

## 6. Stratified Analysis <a id="6-stratified"></a>

In [ ]:
# Define player tiers
player_avgs = played_df.groupby('player_id').agg({
    'pts': 'mean',
    'reb': 'mean',
    'ast': 'mean',
    'minutes': 'mean',
    'game_id': 'count',
    'player_name': 'first',
}).rename(columns={'game_id': 'games'})

player_avgs = player_avgs[player_avgs['games'] >= CONFIG.min_games_played]

def assign_tier(ppg: float) -> str:
    if ppg >= CONFIG.star_threshold:
        return 'Star (20+ PPG)'
    elif ppg >= CONFIG.rotation_threshold:
        return 'Rotation (8-20 PPG)'
    else:
        return 'Bench (<8 PPG)'

player_avgs['tier'] = player_avgs['pts'].apply(assign_tier)

# Tier summary
tier_summary = player_avgs.groupby('tier').agg({
    'player_name': 'count',
    'pts': ['mean', 'std'],
    'minutes': ['mean', 'std'],
}).round(1)

tier_summary.columns = ['Players', 'PPG Mean', 'PPG SD', 'MPG Mean', 'MPG SD']
tier_summary = tier_summary.reindex(['Star (20+ PPG)', 'Rotation (8-20 PPG)', 'Bench (<8 PPG)'])

print("Player Tier Summary:")
print(tier_summary)

In [ ]:
# Merge tiers to game-level data
played_df = played_df.merge(
    player_avgs[['tier']],
    left_on='player_id',
    right_index=True,
    how='left',
    suffixes=('', '_dup')
)

# Drop duplicate tier column if exists
if 'tier_dup' in played_df.columns:
    played_df = played_df.drop(columns=['tier_dup'])

# Variance by tier with clustered bootstrap CIs
# (accounts for repeated measures within players)
print("Performance by Tier (with clustered bootstrap CIs):")
print("-" * 70)

for tier in ['Star (20+ PPG)', 'Rotation (8-20 PPG)', 'Bench (<8 PPG)']:
    tier_data = played_df[played_df['tier'] == tier]
    if len(tier_data) < 100:
        continue
    
    n_players = tier_data['player_id'].nunique()
    n_obs = len(tier_data)
    
    print(f"\n{tier} (n={n_players} players, {n_obs:,} observations):")
    
    for col in ['pts', 'reb', 'ast']:
        mean, ci_lo, ci_hi = clustered_bootstrap_ci(
            tier_data, col, 'player_id', n_bootstrap=CONFIG.n_bootstrap
        )
        cv = tier_data[col].std() / mean if mean > 0 else np.nan
        print(f"  {col.upper()}: {mean:.1f} [{ci_lo:.1f}, {ci_hi:.1f}], CV={cv:.2f}")

print("\n⚠ Note: Clustered bootstrap resamples players (not observations) to account")
print("  for within-player correlation. Standard errors may be wider than naive bootstrap.")

In [ ]:
# Position analysis
if 'start_position' in played_df.columns:
    position_stats = played_df.groupby('start_position').agg({
        'pts': ['mean', 'std', 'count'],
        'reb': ['mean', 'std'],
        'ast': ['mean', 'std'],
    }).round(1)

    position_stats.columns = ['PTS Mean', 'PTS SD', 'N', 'REB Mean', 'REB SD', 'AST Mean', 'AST SD']
    position_stats = position_stats[position_stats['N'] >= 100].sort_values('PTS Mean', ascending=False)

    print("Statistics by Starting Position:")
    print(position_stats)
    print("\nNote: Position strongly influences rebound expectations.")

In [ ]:
# Visualize distributions by tier
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

tier_order = ['Star (20+ PPG)', 'Rotation (8-20 PPG)', 'Bench (<8 PPG)']
colors = ['#2ecc71', '#3498db', '#95a5a6']

for i, col in enumerate(['pts', 'reb', 'ast']):
    for j, tier in enumerate(tier_order):
        data = played_df[played_df['tier'] == tier][col].dropna()
        if len(data) > 0:
            axes[i].hist(data, bins=30, alpha=0.5, label=tier, color=colors[j], density=True)

    axes[i].set_xlabel(col.upper())
    axes[i].set_ylabel('Density')
    axes[i].set_title(f'{col.upper()} by Player Tier')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'tier_distributions.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

---

## 7. Temporal Patterns <a id="7-temporal"></a>

In [ ]:
# Monthly averages
played_df['month'] = played_df['game_date'].dt.to_period('M')

monthly = played_df.groupby('month').agg({
    'pts': ['mean', 'std', 'count'],
    'reb': 'mean',
    'ast': 'mean',
}).round(2)

monthly.columns = ['PTS Mean', 'PTS SD', 'N', 'REB Mean', 'AST Mean']

print("Monthly Performance Averages:")
print(monthly)

In [ ]:
# Visualize temporal trends
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Monthly averages
ax = axes[0]
monthly_plot = monthly.reset_index()
monthly_plot['month_str'] = monthly_plot['month'].astype(str)
ax.plot(monthly_plot['month_str'], monthly_plot['PTS Mean'], marker='o', label='Points')
ax.fill_between(
    monthly_plot['month_str'],
    monthly_plot['PTS Mean'] - monthly_plot['PTS SD']/np.sqrt(monthly_plot['N']),
    monthly_plot['PTS Mean'] + monthly_plot['PTS SD']/np.sqrt(monthly_plot['N']),
    alpha=0.2
)
ax.set_xlabel('Month')
ax.set_ylabel('Points')
ax.set_title('Monthly Scoring Average (±SE)')
ax.tick_params(axis='x', rotation=45)

# Day of week
ax = axes[1]
played_df['dow'] = played_df['game_date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_stats = played_df.groupby('dow')['pts'].agg(['mean', 'std', 'count']).reindex(dow_order)
dow_stats['se'] = dow_stats['std'] / np.sqrt(dow_stats['count'])

ax.bar(dow_order, dow_stats['mean'], yerr=dow_stats['se'], capsize=3, alpha=0.7)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Points')
ax.set_title('Scoring by Day of Week (±SE)')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'temporal_patterns.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

In [ ]:
# Quarter-by-quarter analysis
quarter_cols = ['q1_pts', 'q2_pts', 'q3_pts', 'q4_pts']
quarter_cols = [c for c in quarter_cols if c in played_df.columns]

if len(quarter_cols) == 4:
    # Summary stats
    quarter_summary = played_df[quarter_cols].describe().T[['mean', 'std']].round(2)
    quarter_summary.index = ['Q1', 'Q2', 'Q3', 'Q4']
    print("Quarter Scoring Summary:")
    print(quarter_summary)

    # Inter-quarter correlations
    print("\nInter-Quarter Correlations:")
    q_corr = played_df[quarter_cols].corr().round(2)
    q_corr.index = ['Q1', 'Q2', 'Q3', 'Q4']
    q_corr.columns = ['Q1', 'Q2', 'Q3', 'Q4']
    print(q_corr)
    print("\nObservation: Low inter-quarter correlations (r ≈ 0.2) indicate")
    print("high game-to-game variability within individual games.")

---

## Cross-Season Analysis

The following sections analyze patterns across all 5 NBA seasons (2021-22 through 2025-26) to assess stability of key findings and identify league-wide trends.


In [ ]:
# =============================================================================
# PUBLICATION-QUALITY FIGURE SETTINGS (for Cross-Season Analysis)
# =============================================================================

# Set publication style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

# Colorblind-friendly palette (Okabe-Ito)
COLORS = {
    'orange': '#E69F00',
    'sky_blue': '#56B4E9', 
    'green': '#009E73',
    'yellow': '#F0E442',
    'blue': '#0072B2',
    'vermillion': '#D55E00',
    'purple': '#CC79A7',
    'black': '#000000',
}

TIER_COLORS = {
    'Star (20+ PPG)': COLORS['vermillion'],
    'Rotation (8-20 PPG)': COLORS['blue'],
    'Bench (<8 PPG)': COLORS['sky_blue'],
}

tier_order = ['Star (20+ PPG)', 'Rotation (8-20 PPG)', 'Bench (<8 PPG)']

print("✓ Publication style configured for cross-season analysis")

In [ ]:
# =============================================================================
# CROSS-SEASON ANALYSIS: Explained Variance by Season
# =============================================================================

# Calculate R² (correlation^2) between season avg and actual for each season
from scipy import stats

r2_by_season = []

for season in CONFIG.seasons:
    season_df = played_df[played_df['season'] == season]
    valid = season_df[['season_avg_pts', 'pts']].dropna()
    
    if len(valid) > 100:  # Need enough data
        r, p = stats.pearsonr(valid['season_avg_pts'], valid['pts'])
        r2 = r ** 2
        r2_by_season.append({
            'season': season,
            'r': r,
            'r2': r2,
            'n': len(valid),
            'p_value': p
        })

r2_df = pd.DataFrame(r2_by_season)

# Visualization: R² by Season
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(r2_df['season'], r2_df['r2'], color=COLORS['blue'], edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, r2_val in zip(bars, r2_df['r2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{r2_val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Season', fontsize=12)
ax.set_ylabel('R² (Season Avg vs Actual PTS)', fontsize=12)
ax.set_title('Explained Variance by Season: How Well Does Season Average Predict Single-Game Performance?', fontsize=13)
ax.set_ylim(0, 0.65)
ax.axhline(y=r2_df['r2'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean R² = {r2_df['r2'].mean():.3f}")
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig14_r2_by_season.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

print("\n=== R² BY SEASON ===")
print(r2_df.to_string(index=False))
print(f"\nMean R² across seasons: {r2_df['r2'].mean():.3f}")
print(f"R² stability (SD): {r2_df['r2'].std():.3f}")

In [ ]:
# =============================================================================
# CROSS-SEASON ANALYSIS: Coefficient of Variation by Tier and Season
# =============================================================================

# Calculate CV by tier for each season
cv_by_tier_season = []

for season in CONFIG.seasons:
    season_df = played_df[played_df['season'] == season]
    
    for tier in ['Star (20+ PPG)', 'Rotation (8-20 PPG)', 'Bench (<8 PPG)']:
        tier_df = season_df[season_df['tier'] == tier]
        pts = tier_df['pts'].dropna()
        
        if len(pts) > 50:
            cv = pts.std() / pts.mean() if pts.mean() > 0 else np.nan
            cv_by_tier_season.append({
                'season': season,
                'tier': tier,
                'cv': cv,
                'n': len(pts),
                'mean_pts': pts.mean(),
                'std_pts': pts.std()
            })

cv_df = pd.DataFrame(cv_by_tier_season)

# Visualization: Grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

seasons = CONFIG.seasons
x = np.arange(len(seasons))
width = 0.25

for i, tier in enumerate(['Star (20+ PPG)', 'Rotation (8-20 PPG)', 'Bench (<8 PPG)']):
    tier_data = cv_df[cv_df['tier'] == tier].set_index('season').reindex(seasons)
    offset = (i - 1) * width
    bars = ax.bar(x + offset, tier_data['cv'], width, label=tier, 
                  color=TIER_COLORS[tier], edgecolor='black', linewidth=0.5)

ax.set_xlabel('Season', fontsize=12)
ax.set_ylabel('Coefficient of Variation (CV)', fontsize=12)
ax.set_title('Player Consistency by Tier Across Seasons', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(seasons)
ax.legend(title='Player Tier')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig15_cv_by_season.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

# Print summary table
print("\n=== CV BY TIER AND SEASON ===")
cv_pivot = cv_df.pivot(index='tier', columns='season', values='cv')
print(cv_pivot.round(3).to_string())

In [ ]:
# =============================================================================
# CROSS-SEASON ANALYSIS: Home Advantage Trend
# =============================================================================

# Calculate home win rate by season
home_advantage_by_season = []

for season in CONFIG.seasons:
    season_games = games_df[games_df['game_date'].dt.year.isin([
        int(season.split('-')[0]), int(season.split('-')[0]) + 1
    ])]
    
    # Alternative: match by season string if available
    if 'season' in games_df.columns:
        season_games = games_df[games_df['season'] == season]
    
    if len(season_games) > 50:
        home_wins = (season_games['home_pts'] > season_games['away_pts']).sum()
        total_games = len(season_games)
        home_win_rate = home_wins / total_games
        
        # Wilson confidence interval
        from statsmodels.stats.proportion import proportion_confint
        ci_low, ci_high = proportion_confint(home_wins, total_games, alpha=0.05, method='wilson')
        
        home_advantage_by_season.append({
            'season': season,
            'home_win_rate': home_win_rate,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'n_games': total_games,
            'home_wins': home_wins
        })

home_df = pd.DataFrame(home_advantage_by_season)

# Visualization: Home win rate trend with CI
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(home_df['season'], home_df['home_win_rate'], 'o-', color=COLORS['blue'], 
        linewidth=2, markersize=10, label='Home Win Rate')
ax.fill_between(home_df['season'], home_df['ci_low'], home_df['ci_high'], 
                alpha=0.3, color=COLORS['blue'], label='95% CI')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='No Advantage (50%)')

ax.set_xlabel('Season', fontsize=12)
ax.set_ylabel('Home Win Rate', fontsize=12)
ax.set_title('Home Court Advantage Trend Across Seasons', fontsize=13)
ax.set_ylim(0.45, 0.65)
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig16_home_trend.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

print("\n=== HOME ADVANTAGE BY SEASON ===")
print(home_df.to_string(index=False))
print(f"\nOverall home win rate: {home_df['home_win_rate'].mean():.1%}")

In [ ]:
# =============================================================================
# CROSS-SEASON ANALYSIS: League Evolution Trends
# =============================================================================

# Calculate league-wide metrics by season
league_trends = []

for season in CONFIG.seasons:
    season_df = played_df[played_df['season'] == season]
    
    if len(season_df) > 100:
        # Calculate key metrics
        metrics = {
            'season': season,
            'mean_pts': season_df['pts'].mean(),
            'mean_3pa': season_df['fg3a'].mean() if 'fg3a' in season_df.columns else np.nan,
            'mean_3p_rate': (season_df['fg3a'] / season_df['fga']).mean() if 'fga' in season_df.columns and 'fg3a' in season_df.columns else np.nan,
            'mean_ts_pct': season_df['ts_pct'].mean() if 'ts_pct' in season_df.columns else np.nan,
            'mean_pace': season_df.get('pace', pd.Series([np.nan])).mean() if 'pace' in season_df.columns else np.nan,
            'variance_pts': season_df['pts'].var(),
            'n_games': season_df['game_id'].nunique()
        }
        league_trends.append(metrics)

league_df = pd.DataFrame(league_trends)

# Visualization: 3-panel league evolution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Points per game
ax = axes[0]
ax.plot(league_df['season'], league_df['mean_pts'], 'o-', color=COLORS['blue'], linewidth=2, markersize=10)
ax.set_xlabel('Season', fontsize=11)
ax.set_ylabel('Mean Points', fontsize=11)
ax.set_title('Average Points per Player-Game', fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel 2: 3-point rate (if available)
ax = axes[1]
if not league_df['mean_3p_rate'].isna().all():
    ax.plot(league_df['season'], league_df['mean_3p_rate'], 'o-', color=COLORS['orange'], linewidth=2, markersize=10)
    ax.set_ylabel('3PA / FGA Rate', fontsize=11)
    ax.set_title('3-Point Attempt Rate', fontsize=12)
else:
    ax.text(0.5, 0.5, 'Data Not Available', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('Season', fontsize=11)
ax.tick_params(axis='x', rotation=45)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel 3: True Shooting % (if available)
ax = axes[2]
if not league_df['mean_ts_pct'].isna().all():
    ax.plot(league_df['season'], league_df['mean_ts_pct'], 'o-', color=COLORS['green'], linewidth=2, markersize=10)
    ax.set_ylabel('True Shooting %', fontsize=11)
    ax.set_title('League-Wide Efficiency', fontsize=12)
else:
    ax.text(0.5, 0.5, 'Data Not Available', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('Season', fontsize=11)
ax.tick_params(axis='x', rotation=45)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('NBA League Evolution: 2021-22 to 2025-26', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig11_league_evolution.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

print("\n=== LEAGUE EVOLUTION METRICS ===")
print(league_df.to_string(index=False))

In [ ]:
# =============================================================================
# DATA QUALITY: Missing Data Visualization by Season
# =============================================================================

# Calculate missing data percentage by column group and season
col_groups = {
    'Boxscore': ['pts', 'reb', 'ast', 'stl', 'blk', 'tov', 'pf', 'fg_pct', 'fg3_pct', 'ft_pct', 'min'],
    'Advanced': ['ts_pct', 'usg_pct', 'off_rating', 'def_rating', 'net_rating', 'pie'],
    'Pregame (Player)': [c for c in df.columns if c.startswith(('season_avg', 'last5_avg', 'last10_'))],
    'Pregame (Team)': [c for c in df.columns if c.startswith('team_pre_')],
}

# Filter to columns that exist
col_groups = {k: [c for c in v if c in df.columns] for k, v in col_groups.items()}

missing_by_season_group = []

for season in CONFIG.seasons:
    season_df = df[df['season'] == season]
    
    for group, cols in col_groups.items():
        if cols:
            missing_pct = season_df[cols].isna().mean().mean() * 100
            missing_by_season_group.append({
                'season': season,
                'group': group,
                'missing_pct': missing_pct
            })

missing_df = pd.DataFrame(missing_by_season_group)

# Pivot for heatmap
missing_pivot = missing_df.pivot(index='group', columns='season', values='missing_pct')

# Visualization: Heatmap
fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(missing_pivot, annot=True, fmt='.1f', cmap='YlOrRd', 
            cbar_kws={'label': 'Missing %'}, ax=ax)
ax.set_title('Missing Data by Feature Group and Season', fontsize=13)
ax.set_xlabel('Season', fontsize=11)
ax.set_ylabel('Feature Group', fontsize=11)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig10_missing_data.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

print("\n=== MISSING DATA SUMMARY ===")
print(missing_pivot.round(1).to_string())

In [ ]:
# =============================================================================
# CROSS-SEASON ANALYSIS: When Does Player Rolling Average Stabilize?
# =============================================================================

# For a sample of star players, track how their rolling average stabilizes over games
import warnings
warnings.filterwarnings('ignore')

# Get star players with enough games across multiple seasons
star_players = played_df[played_df['tier'] == 'Star (20+ PPG)']['player_name'].value_counts()
sample_stars = star_players[star_players >= 100].head(10).index.tolist()

stabilization_data = []

for player in sample_stars:
    player_df = played_df[played_df['player_name'] == player].sort_values('game_date').reset_index(drop=True)
    
    if len(player_df) >= 30:
        # Calculate cumulative mean and track deviation from season avg
        for i in range(1, min(len(player_df) + 1, 82)):  # First 82 games
            rolling_mean = player_df['pts'].iloc[:i].mean()
            season_avg = player_df['pts'].mean()
            deviation_pct = abs(rolling_mean - season_avg) / season_avg * 100 if season_avg > 0 else 0
            
            stabilization_data.append({
                'player': player,
                'game_num': i,
                'rolling_mean': rolling_mean,
                'season_avg': season_avg,
                'deviation_pct': deviation_pct
            })

stab_df = pd.DataFrame(stabilization_data)

# Calculate mean deviation across players at each game number
mean_deviation = stab_df.groupby('game_num')['deviation_pct'].agg(['mean', 'std']).reset_index()

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Individual player trajectories
ax = axes[0]
for player in sample_stars[:5]:  # Top 5 for clarity
    player_data = stab_df[stab_df['player'] == player]
    ax.plot(player_data['game_num'], player_data['deviation_pct'], alpha=0.7, label=player.split()[-1])

ax.axhline(y=5, color='red', linestyle='--', linewidth=1, label='5% threshold')
ax.set_xlabel('Games Played', fontsize=11)
ax.set_ylabel('Deviation from Season Avg (%)', fontsize=11)
ax.set_title('Rolling Average Stabilization: Individual Stars', fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(1, 50)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel 2: Average across all sampled players
ax = axes[1]
ax.plot(mean_deviation['game_num'], mean_deviation['mean'], 'b-', linewidth=2, label='Mean Deviation')
ax.fill_between(mean_deviation['game_num'], 
                mean_deviation['mean'] - mean_deviation['std'],
                mean_deviation['mean'] + mean_deviation['std'],
                alpha=0.3, color='blue', label='±1 SD')
ax.axhline(y=5, color='red', linestyle='--', linewidth=1, label='5% threshold')

# Find stabilization point (first game where mean deviation < 5%)
stable_point = mean_deviation[mean_deviation['mean'] < 5]['game_num'].min()
if not pd.isna(stable_point):
    ax.axvline(x=stable_point, color='green', linestyle=':', linewidth=2, 
               label=f'Stabilizes at game {int(stable_point)}')

ax.set_xlabel('Games Played', fontsize=11)
ax.set_ylabel('Mean Deviation from Season Avg (%)', fontsize=11)
ax.set_title('Rolling Average Stabilization: League-Wide Pattern', fontsize=12)
ax.legend(loc='upper right')
ax.set_xlim(1, 50)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('When Does a Player\'s Rolling Average Stabilize?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig12_season_stabilization.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

print(f"\n=== STABILIZATION ANALYSIS ===")
print(f"Sample size: {len(sample_stars)} star players")
if not pd.isna(stable_point):
    print(f"Rolling average stabilizes (within 5% of true avg) after ~{int(stable_point)} games")

In [ ]:
# =============================================================================
# CROSS-SEASON ANALYSIS: Player Case Studies
# =============================================================================

# Define case study players (manually selected for interesting trajectories)
case_study_candidates = {
    'Consistent Star': ['Giannis Antetokounmpo', 'Stephen Curry', 'Kevin Durant', 'LeBron James'],
    'Breakout Player': ['Tyrese Maxey', 'Jayson Tatum', 'Anthony Edwards', 'Ja Morant'],
    'Declining Player': ['Chris Paul', 'Carmelo Anthony', 'Russell Westbrook'],
    'High Variance': ['Russell Westbrook', 'Anthony Davis', 'Zion Williamson', 'Kawhi Leonard']
}

# Find available players in our dataset
available_players = played_df['player_name'].unique()

case_studies = {}
for category, candidates in case_study_candidates.items():
    for player in candidates:
        if player in available_players:
            player_data = played_df[played_df['player_name'] == player]
            if len(player_data) >= 50:  # Need enough data
                case_studies[player] = {
                    'category': category,
                    'n_games': len(player_data),
                    'seasons': player_data['season'].unique().tolist()
                }
                break

print("=== CASE STUDY PLAYERS ===")
for player, info in case_studies.items():
    print(f"{info['category']}: {player} ({info['n_games']} games, {len(info['seasons'])} seasons)")

# Calculate season-by-season stats for each case study
case_stats = []
for player in case_studies.keys():
    player_df = played_df[played_df['player_name'] == player]
    
    for season in player_df['season'].unique():
        season_df = player_df[player_df['season'] == season]
        pts = season_df['pts']
        
        case_stats.append({
            'player': player,
            'category': case_studies[player]['category'],
            'season': season,
            'mean_pts': pts.mean(),
            'std_pts': pts.std(),
            'cv': pts.std() / pts.mean() if pts.mean() > 0 else np.nan,
            'n_games': len(season_df)
        })

case_df = pd.DataFrame(case_stats)

# Visualization: 2x2 grid for case studies
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

categories = list(case_study_candidates.keys())[:4]
for idx, (ax, category) in enumerate(zip(axes.flat, categories)):
    # Find player in this category
    cat_players = [p for p, info in case_studies.items() if info['category'] == category]
    
    if cat_players:
        player = cat_players[0]
        player_data = case_df[case_df['player'] == player]
        
        # Points with error bars (SD)
        ax.errorbar(player_data['season'], player_data['mean_pts'], 
                   yerr=player_data['std_pts'], fmt='o-', capsize=5,
                   linewidth=2, markersize=10, label='Mean ± SD')
        
        # Add CV annotation
        for _, row in player_data.iterrows():
            ax.annotate(f'CV={row["cv"]:.2f}', 
                       xy=(row['season'], row['mean_pts'] + row['std_pts'] + 2),
                       ha='center', fontsize=9)
        
        ax.set_xlabel('Season', fontsize=11)
        ax.set_ylabel('Points per Game', fontsize=11)
        ax.set_title(f'{category}: {player}', fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    else:
        ax.text(0.5, 0.5, f'{category}\nNo player found', ha='center', va='center', 
                transform=ax.transAxes, fontsize=12)

plt.suptitle('Player Case Studies: Performance Trajectories Across Seasons', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig13_case_studies.png', dpi=CONFIG.fig_dpi, bbox_inches='tight')
plt.show()

print("\n=== CASE STUDY STATISTICS ===")
print(case_df.to_string(index=False))

---

## 8. Publication-Quality Visualizations <a id="8-visualizations"></a>

This section generates standalone, high-quality figures for presentations and reports.

# =============================================================================
# PUBLICATION-QUALITY FIGURE SETTINGS
# =============================================================================

# Set publication style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

# Colorblind-friendly palette (Okabe-Ito)
COLORS = {
    'orange': '#E69F00',
    'sky_blue': '#56B4E9', 
    'green': '#009E73',
    'yellow': '#F0E442',
    'blue': '#0072B2',
    'vermillion': '#D55E00',
    'purple': '#CC79A7',
    'black': '#000000',
}

TIER_COLORS = {
    'Star (20+ PPG)': COLORS['vermillion'],
    'Rotation (8-20 PPG)': COLORS['blue'],
    'Bench (<8 PPG)': COLORS['sky_blue'],
}

print("✓ Publication style configured")
print(f"✓ Figure output: {CONFIG.fig_dir.absolute()}")

In [ ]:
# =============================================================================
# FIGURE 1: Violin Plots - Performance by Player Tier
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))

tier_order = ['Star (20+ PPG)', 'Rotation (8-20 PPG)', 'Bench (<8 PPG)']
palette = [TIER_COLORS[t] for t in tier_order]

for i, (col, label, unit) in enumerate([
    ('pts', 'Points', 'per game'),
    ('reb', 'Rebounds', 'per game'), 
    ('ast', 'Assists', 'per game')
]):
    ax = axes[i]
    
    # Create violin plot
    parts = ax.violinplot(
        [played_df[played_df['tier'] == tier][col].dropna() for tier in tier_order],
        positions=[1, 2, 3],
        showmeans=True,
        showmedians=True,
        widths=0.8
    )
    
    # Color the violins
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(palette[j])
        pc.set_alpha(0.7)
    
    # Style lines
    parts['cmeans'].set_color('black')
    parts['cmeans'].set_linewidth(2)
    parts['cmedians'].set_color('white')
    parts['cmedians'].set_linewidth(1.5)
    
    # Add individual points (subsampled for clarity)
    for j, tier in enumerate(tier_order):
        data = played_df[played_df['tier'] == tier][col].dropna()
        sample = data.sample(min(200, len(data)), random_state=42)
        jitter = np.random.normal(0, 0.08, len(sample))
        ax.scatter(j + 1 + jitter, sample, alpha=0.15, s=3, color='black')
    
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Stars', 'Rotation', 'Bench'], fontsize=9)
    ax.set_ylabel(f'{label} ({unit})')
    ax.set_title(f'{label} Distribution by Tier', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0.3, 3.7)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig1_tier_violins.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig1_tier_violins.png")

In [ ]:
# =============================================================================
# FIGURE 2: Prediction Accuracy - Season Average vs Actual (Joint Plot Style)
# =============================================================================

fig = plt.figure(figsize=(10, 10))

# Create grid for main plot and marginals
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                      hspace=0.05, wspace=0.05)

ax_main = fig.add_subplot(gs[1, 0])
ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

# Prepare data
valid = played_df[['season_avg_pts', 'pts', 'tier']].dropna()

# Main scatter with tier coloring
for tier in tier_order:
    tier_data = valid[valid['tier'] == tier]
    sample = tier_data.sample(min(1000, len(tier_data)), random_state=42)
    ax_main.scatter(sample['season_avg_pts'], sample['pts'], 
                   alpha=0.3, s=15, c=TIER_COLORS[tier], label=tier, edgecolors='none')

# Add identity line
max_val = max(valid['season_avg_pts'].max(), valid['pts'].max())
ax_main.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, linewidth=1.5, label='Perfect prediction')

# Add regression line
z = np.polyfit(valid['season_avg_pts'], valid['pts'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, valid['season_avg_pts'].max(), 100)
ax_main.plot(x_line, p(x_line), color=COLORS['vermillion'], linewidth=2, 
            label=f'Fit: y = {z[0]:.2f}x + {z[1]:.2f}')

ax_main.set_xlabel('Season Average Points', fontsize=11)
ax_main.set_ylabel('Game Points', fontsize=11)
ax_main.legend(loc='upper left', fontsize=8, framealpha=0.9)
ax_main.set_xlim(-1, 38)
ax_main.set_ylim(-1, 65)

# Top marginal (season avg distribution)
for tier in tier_order:
    tier_data = valid[valid['tier'] == tier]['season_avg_pts']
    ax_top.hist(tier_data, bins=40, alpha=0.5, color=TIER_COLORS[tier], density=True)
ax_top.set_ylabel('Density')
ax_top.tick_params(labelbottom=False)
ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)

# Right marginal (actual pts distribution)  
for tier in tier_order:
    tier_data = valid[valid['tier'] == tier]['pts']
    ax_right.hist(tier_data, bins=40, alpha=0.5, color=TIER_COLORS[tier], 
                 density=True, orientation='horizontal')
ax_right.set_xlabel('Density')
ax_right.tick_params(labelleft=False)
ax_right.spines['top'].set_visible(False)
ax_right.spines['right'].set_visible(False)

# Add R² annotation
r2 = valid['season_avg_pts'].corr(valid['pts']) ** 2
ax_main.text(0.95, 0.05, f'R² = {r2:.3f}', transform=ax_main.transAxes,
            fontsize=12, fontweight='bold', ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.savefig(CONFIG.fig_dir / 'fig2_prediction_joint.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig2_prediction_joint.png")

In [ ]:
# =============================================================================
# FIGURE 3: Position-Based Performance Expectations
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))

positions = ['G', 'F', 'C']
pos_labels = ['Guards', 'Forwards', 'Centers']
pos_colors = [COLORS['blue'], COLORS['orange'], COLORS['green']]

for i, (col, ylabel, title) in enumerate([
    ('pts', 'Points per game', 'Scoring by Position'),
    ('reb', 'Rebounds per game', 'Rebounding by Position'),
    ('ast', 'Assists per game', 'Assists by Position')
]):
    ax = axes[i]
    
    # Box plot with individual points
    pos_data = [played_df[played_df['start_position'] == p][col].dropna() for p in positions]
    
    bp = ax.boxplot(pos_data, positions=[1, 2, 3], widths=0.6, patch_artist=True,
                   showfliers=False, medianprops=dict(color='white', linewidth=2))
    
    # Color boxes
    for patch, color in zip(bp['boxes'], pos_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add swarm of points
    for j, (p, color) in enumerate(zip(positions, pos_colors)):
        data = played_df[played_df['start_position'] == p][col].dropna()
        sample = data.sample(min(150, len(data)), random_state=42)
        jitter = np.random.normal(0, 0.1, len(sample))
        ax.scatter(j + 1 + jitter, sample, alpha=0.2, s=8, color='black', zorder=0)
    
    # Add mean markers
    means = [d.mean() for d in pos_data]
    ax.scatter([1, 2, 3], means, color='black', s=50, marker='D', zorder=5, label='Mean')
    
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(pos_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    if i == 0:
        ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig3_position_performance.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig3_position_performance.png")

In [ ]:
# =============================================================================
# FIGURE 4: Player Consistency - Coefficient of Variation by Tier
# =============================================================================

fig, ax = plt.subplots(figsize=(8, 5))

# Calculate CV for each player with enough games
player_cv = played_df.groupby(['player_id', 'tier']).agg({
    'pts': ['mean', 'std', 'count'],
    'player_name': 'first'
}).reset_index()
player_cv.columns = ['player_id', 'tier', 'pts_mean', 'pts_std', 'games', 'player_name']
player_cv = player_cv[player_cv['games'] >= 20]  # At least 20 games
player_cv['cv'] = player_cv['pts_std'] / player_cv['pts_mean']
player_cv = player_cv.dropna()

# Create grouped data
cv_by_tier = []
for tier in tier_order:
    tier_cv = player_cv[player_cv['tier'] == tier]['cv']
    cv_by_tier.append(tier_cv)

# Violin + strip plot
parts = ax.violinplot(cv_by_tier, positions=[1, 2, 3], widths=0.7, 
                      showmeans=True, showmedians=False)

for j, pc in enumerate(parts['bodies']):
    pc.set_facecolor([TIER_COLORS[t] for t in tier_order][j])
    pc.set_alpha(0.6)

parts['cmeans'].set_color('black')
parts['cmeans'].set_linewidth(2)

# Add individual points
for j, (tier, data) in enumerate(zip(tier_order, cv_by_tier)):
    jitter = np.random.normal(0, 0.08, len(data))
    ax.scatter(j + 1 + jitter, data, alpha=0.4, s=20, 
              c=TIER_COLORS[tier], edgecolors='white', linewidth=0.5)

# Highlight most/least consistent
for tier_idx, tier in enumerate(tier_order):
    tier_data = player_cv[player_cv['tier'] == tier].nsmallest(1, 'cv')
    if len(tier_data) > 0:
        row = tier_data.iloc[0]
        ax.annotate(row['player_name'].split()[-1], 
                   xy=(tier_idx + 1, row['cv']),
                   xytext=(tier_idx + 1.3, row['cv'] - 0.05),
                   fontsize=7, alpha=0.7,
                   arrowprops=dict(arrowstyle='-', alpha=0.3))

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Stars\n(20+ PPG)', 'Rotation\n(8-20 PPG)', 'Bench\n(<8 PPG)'])
ax.set_ylabel('Coefficient of Variation (SD / Mean)')
ax.set_title('Player Consistency: Lower CV = More Predictable', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add annotation
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(3.4, 0.51, 'CV = 0.5', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig4_player_consistency.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig4_player_consistency.png")

In [ ]:
# =============================================================================
# FIGURE 5: Home Court Advantage Analysis
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Panel A: Win rate with CI
ax = axes[0]
home_win_pct = games_df['home_win'].mean() * 100
ci_lo, ci_hi = proportion_confint(games_df['home_win'].sum(), len(games_df), 
                                   alpha=0.05, method='wilson')

bars = ax.bar(['Away', 'Home'], [100 - home_win_pct, home_win_pct], 
              color=[COLORS['sky_blue'], COLORS['vermillion']], alpha=0.8, width=0.6)
ax.errorbar([1], [home_win_pct], yerr=[[home_win_pct - ci_lo*100], [ci_hi*100 - home_win_pct]],
           fmt='none', color='black', capsize=5, capthick=2)
ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Win Rate (%)')
ax.set_title('Home Court Win Rate', fontweight='bold')
ax.set_ylim(0, 70)
ax.text(1, home_win_pct + 5, f'{home_win_pct:.1f}%', ha='center', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel B: Point differential distribution
ax = axes[1]
ax.hist(games_df['point_diff'], bins=35, color=COLORS['blue'], alpha=0.7, edgecolor='white')
ax.axvline(0, color='black', linestyle='-', linewidth=1.5)
ax.axvline(games_df['point_diff'].mean(), color=COLORS['vermillion'], 
          linestyle='--', linewidth=2, label=f"Mean: {games_df['point_diff'].mean():+.1f}")
ax.set_xlabel('Point Differential (Home − Away)')
ax.set_ylabel('Number of Games')
ax.set_title('Home Point Differential', fontweight='bold')
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel C: Total points distribution
ax = axes[2]
ax.hist(games_df['total_pts'], bins=30, color=COLORS['green'], alpha=0.7, edgecolor='white')
mean_total = games_df['total_pts'].mean()
std_total = games_df['total_pts'].std()
ax.axvline(mean_total, color=COLORS['vermillion'], linestyle='--', linewidth=2)
ax.axvspan(mean_total - std_total, mean_total + std_total, alpha=0.15, color=COLORS['vermillion'])
ax.set_xlabel('Total Points (Both Teams)')
ax.set_ylabel('Number of Games')
ax.set_title(f'Game Totals: {mean_total:.0f} ± {std_total:.0f}', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig5_home_advantage.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig5_home_advantage.png")

In [ ]:
# =============================================================================
# FIGURE 6: Quarter-by-Quarter Performance Flow
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Quarter means by tier
ax = axes[0]
quarter_cols = ['q1_pts', 'q2_pts', 'q3_pts', 'q4_pts']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

for tier in tier_order:
    tier_data = played_df[played_df['tier'] == tier]
    means = [tier_data[q].mean() for q in quarter_cols]
    sems = [tier_data[q].std() / np.sqrt(len(tier_data)) for q in quarter_cols]
    
    ax.errorbar(quarters, means, yerr=sems, marker='o', markersize=8,
               capsize=4, linewidth=2, label=tier, color=TIER_COLORS[tier])

ax.set_xlabel('Quarter')
ax.set_ylabel('Points per Quarter')
ax.set_title('Quarter Scoring by Player Tier', fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, None)

# Panel B: Inter-quarter correlation heatmap
ax = axes[1]
q_corr = played_df[quarter_cols].corr()
q_corr.index = quarters
q_corr.columns = quarters

im = ax.imshow(q_corr, cmap='RdYlBu_r', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(quarters)
ax.set_yticklabels(quarters)

# Add correlation values
for i in range(4):
    for j in range(4):
        text = ax.text(j, i, f'{q_corr.iloc[i, j]:.2f}',
                      ha='center', va='center', color='black', fontsize=11, fontweight='bold')

ax.set_title('Inter-Quarter Correlations', fontweight='bold')
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Correlation (r)', rotation=270, labelpad=15)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig6_quarter_analysis.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig6_quarter_analysis.png")

In [ ]:
# =============================================================================
# FIGURE 7: Feature Correlation Heatmap (Clustered)
# =============================================================================

# Select key features for correlation analysis
feature_cols = [
    'pts', 'reb', 'ast', 'stl', 'blk', 'tov',
    'minutes', 'fg_pct', 'ts_pct', 'usg_pct', 
    'plus_minus', 'pie'
]
feature_cols = [c for c in feature_cols if c in played_df.columns]

# Compute correlation matrix
corr_data = played_df[feature_cols].dropna()
corr_matrix = corr_data.corr()

# Hierarchical clustering for ordering
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

# Convert correlation to distance and cluster
dist_matrix = 1 - np.abs(corr_matrix)
np.fill_diagonal(dist_matrix.values, 0)
linkage_matrix = linkage(squareform(dist_matrix), method='ward')
order = leaves_list(linkage_matrix)

# Reorder correlation matrix
ordered_cols = [feature_cols[i] for i in order]
corr_ordered = corr_matrix.loc[ordered_cols, ordered_cols]

# Create figure
fig, ax = plt.subplots(figsize=(9, 8))

# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_ordered, dtype=bool), k=1)

# Plot heatmap
im = ax.imshow(corr_ordered.where(~mask), cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')

# Set ticks
ax.set_xticks(range(len(ordered_cols)))
ax.set_yticks(range(len(ordered_cols)))
ax.set_xticklabels(ordered_cols, rotation=45, ha='right')
ax.set_yticklabels(ordered_cols)

# Add correlation values (lower triangle only)
for i in range(len(ordered_cols)):
    for j in range(len(ordered_cols)):
        if i > j:  # Lower triangle only
            val = corr_ordered.iloc[i, j]
            color = 'white' if abs(val) > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', 
                   color=color, fontsize=8)

ax.set_title('Feature Correlations (Hierarchically Clustered)', fontweight='bold', pad=10)

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Pearson Correlation', rotation=270, labelpad=15)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig7_feature_correlations.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig7_feature_correlations.png")

In [ ]:
# =============================================================================
# FIGURE 8: Prediction Residual Analysis
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# Calculate residuals (actual - predicted using season avg)
valid = played_df[['pts', 'season_avg_pts', 'tier', 'minutes']].dropna()
valid['residual'] = valid['pts'] - valid['season_avg_pts']
valid['abs_residual'] = np.abs(valid['residual'])

# Panel A: Residual distribution by tier
ax = axes[0, 0]
for tier in tier_order:
    tier_resid = valid[valid['tier'] == tier]['residual']
    ax.hist(tier_resid, bins=40, alpha=0.5, label=tier, 
           color=TIER_COLORS[tier], density=True)

ax.axvline(0, color='black', linestyle='-', linewidth=1.5)
ax.set_xlabel('Residual (Actual − Season Average)')
ax.set_ylabel('Density')
ax.set_title('Prediction Residuals by Tier', fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel B: Residual vs predicted (heteroscedasticity check)
ax = axes[0, 1]
sample = valid.sample(min(3000, len(valid)), random_state=42)
for tier in tier_order:
    tier_data = sample[sample['tier'] == tier]
    ax.scatter(tier_data['season_avg_pts'], tier_data['residual'], 
              alpha=0.3, s=10, c=TIER_COLORS[tier], label=tier)

ax.axhline(0, color='black', linestyle='-', linewidth=1.5)
ax.set_xlabel('Season Average Points (Predicted)')
ax.set_ylabel('Residual')
ax.set_title('Residuals vs Predicted: Heteroscedasticity Check', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add LOWESS trend line
from scipy.ndimage import uniform_filter1d
sorted_idx = np.argsort(sample['season_avg_pts'])
x_sorted = sample['season_avg_pts'].values[sorted_idx]
y_sorted = sample['residual'].values[sorted_idx]
y_smooth = uniform_filter1d(y_sorted, size=200)
ax.plot(x_sorted, y_smooth, color='black', linewidth=2, label='Trend')

# Panel C: Absolute residual by minutes played
ax = axes[1, 0]
minutes_bins = pd.cut(valid['minutes'], bins=[0, 10, 20, 30, 40, 55], 
                      labels=['0-10', '10-20', '20-30', '30-40', '40+'])
abs_resid_by_min = valid.groupby(minutes_bins)['abs_residual'].agg(['mean', 'std', 'count'])
abs_resid_by_min['se'] = abs_resid_by_min['std'] / np.sqrt(abs_resid_by_min['count'])

x_pos = range(len(abs_resid_by_min))
ax.bar(x_pos, abs_resid_by_min['mean'], yerr=abs_resid_by_min['se'], 
      capsize=4, color=COLORS['blue'], alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(abs_resid_by_min.index)
ax.set_xlabel('Minutes Played')
ax.set_ylabel('Mean Absolute Error')
ax.set_title('Prediction Error by Playing Time', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel D: Error distribution summary
ax = axes[1, 1]
error_stats = []
for tier in tier_order:
    tier_resid = valid[valid['tier'] == tier]['abs_residual']
    error_stats.append({
        'tier': tier.split()[0],  # Short name
        'mae': tier_resid.mean(),
        'median_ae': tier_resid.median(),
        'p90': tier_resid.quantile(0.9),
    })

error_df = pd.DataFrame(error_stats)
x = np.arange(len(tier_order))
width = 0.25

bars1 = ax.bar(x - width, error_df['mae'], width, label='MAE', color=COLORS['blue'], alpha=0.8)
bars2 = ax.bar(x, error_df['median_ae'], width, label='Median AE', color=COLORS['green'], alpha=0.8)
bars3 = ax.bar(x + width, error_df['p90'], width, label='90th %ile', color=COLORS['vermillion'], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(['Stars', 'Rotation', 'Bench'])
ax.set_ylabel('Absolute Error (Points)')
ax.set_title('Error Metrics by Tier', fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig8_residual_analysis.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig8_residual_analysis.png")

In [ ]:
# =============================================================================
# FIGURE 9: Feature Importance (Correlation-based) for Player Props
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Features to analyze
predictor_features = [
    'season_avg_pts', 'last5_avg_pts', 'season_avg_min', 'last5_avg_min',
    'season_avg_usg_pct', 'season_avg_ts_pct', 'team_pre_season_ppg',
    'team_pre_days_rest', 'team_pre_is_home', 'minutes'
]
predictor_features = [f for f in predictor_features if f in played_df.columns]

# Clean feature names for display
clean_names = {
    'season_avg_pts': 'Season Avg PTS',
    'last5_avg_pts': 'Last 5 Avg PTS',
    'season_avg_min': 'Season Avg MIN',
    'last5_avg_min': 'Last 5 Avg MIN',
    'season_avg_usg_pct': 'Season USG%',
    'season_avg_ts_pct': 'Season TS%',
    'team_pre_season_ppg': 'Team PPG',
    'team_pre_days_rest': 'Days Rest',
    'team_pre_is_home': 'Home Game',
    'minutes': 'Minutes (actual)',
}

for idx, (target, title) in enumerate([('pts', 'Points'), ('reb', 'Rebounds'), ('ast', 'Assists')]):
    ax = axes[idx]
    
    # Calculate correlations
    correlations = []
    for feat in predictor_features:
        valid = played_df[[target, feat]].dropna()
        if len(valid) > 100:
            r = valid[target].corr(valid[feat])
            correlations.append({
                'feature': clean_names.get(feat, feat),
                'r': r,
                'abs_r': abs(r)
            })
    
    # Sort by absolute correlation
    corr_df = pd.DataFrame(correlations).sort_values('abs_r', ascending=True)
    
    # Create horizontal bar chart
    colors = [COLORS['vermillion'] if r > 0 else COLORS['blue'] for r in corr_df['r']]
    bars = ax.barh(range(len(corr_df)), corr_df['r'], color=colors, alpha=0.7)
    
    ax.set_yticks(range(len(corr_df)))
    ax.set_yticklabels(corr_df['feature'], fontsize=9)
    ax.set_xlabel('Correlation with ' + title)
    ax.set_title(f'{title} Predictors', fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-0.2, 0.85)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add value labels
    for bar, val in zip(bars, corr_df['r']):
        x_pos = val + 0.02 if val > 0 else val - 0.02
        ha = 'left' if val > 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height()/2, 
               f'{val:.2f}', va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.savefig(CONFIG.fig_dir / 'fig9_feature_importance.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: fig9_feature_importance.png")

In [ ]:
# =============================================================================
# FIGURE SUMMARY
# =============================================================================

print("=" * 60)
print("PUBLICATION-QUALITY FIGURES GENERATED")
print("=" * 60)

figures = [
    ('fig1_tier_violins.png', 'Performance distributions by player tier'),
    ('fig2_prediction_joint.png', 'Season avg vs actual (joint plot with marginals)'),
    ('fig3_position_performance.png', 'Position-based performance expectations'),
    ('fig4_player_consistency.png', 'Player consistency (CV) by tier'),
    ('fig5_home_advantage.png', 'Home court advantage analysis'),
    ('fig6_quarter_analysis.png', 'Quarter-by-quarter patterns'),
    ('fig7_feature_correlations.png', 'Hierarchically clustered feature correlations'),
    ('fig8_residual_analysis.png', 'Prediction residual diagnostics'),
    ('fig9_feature_importance.png', 'Feature importance for player props'),
]

print(f"\nOutput directory: {CONFIG.fig_dir.absolute()}\n")
for fname, desc in figures:
    fpath = CONFIG.fig_dir / fname
    exists = "✓" if fpath.exists() else "✗"
    print(f"  {exists} {fname}")
    print(f"      {desc}")
    print()

print("-" * 60)
print("All figures use:")
print("  • Colorblind-friendly Okabe-Ito palette")
print("  • 300 DPI resolution")
print("  • Clean white backgrounds")
print("  • Publication-ready typography")
print("-" * 60)

---

## 9. Summary & Recommendations <a id="9-summary"></a>

In [ ]:
summary = """
## EDA Summary

### Dataset Characteristics
- **Size:** {n_rows:,} player-game observations, {n_cols} features
- **Seasons:** {seasons}
- **Players:** {n_players:,} unique players
- **Games:** {n_games:,} complete games

### Data Quality
- Pregame averages missing for first games (structurally missing - MNAR)
- Tracking stats (speed, distance) have ~15-20% missing (MAR)
- No significant data integrity issues found
- Outliers are legitimate extreme performances, not errors

### Key Distributional Properties
- **Target variables (PTS, REB, AST):** Right-skewed, non-normal
- **Points:** Mean ~10, high variance (CV ≈ 0.8-1.0)
- **Rebounds/Assists:** Even more variable (CV > 1.0)
- All targets have substantial tail probabilities
- Note: Non-normality does not preclude CLT-based inference on means

### Bivariate Relationships
- Season averages correlate strongly with game performance (r ≈ 0.6-0.7)
- Minutes correlate with scoring (r ≈ 0.6)
- Usage rate correlates with points (r ≈ 0.4)
- All correlations significant after FDR correction

### Stratification Effects
- Stars (20+ PPG): Lower CV, more consistent performance
- Bench players: Higher CV, less consistent
- Position strongly affects rebound expectations

### Temporal Patterns
- No strong monthly trends detected
- Low inter-quarter correlation (r ≈ 0.2)
- Home court advantage: ~54-56% win rate (Wilson CI), ~2-3 point differential

### Statistical Methods Applied
- Bootstrap CIs for means and correlations
- Clustered bootstrap for player-level aggregates (accounts for repeated measures)
- FDR correction (Benjamini-Hochberg) for multiple correlation tests
- Wilson score intervals for proportions
- Cohen's h effect size for home advantage

### Recommendations for Modeling
1. **Handle non-normality:** Consider quantile regression or robust methods
2. **Stratify by tier:** Build separate models for stars vs bench
3. **Position features:** Include position for rebounds/blocks
4. **Missing data:** Impute first-game averages with league/position means
5. **Validation:** Use time-series split (not random) for any prediction task
6. **Clustering:** Account for player-level clustering in standard errors
""".format(
    n_rows=len(df),
    n_cols=len(df.columns),
    seasons=', '.join(CONFIG.seasons),
    n_players=df['player_id'].nunique(),
    n_games=len(games_df),
)

print(summary)

In [ ]:
# Final statistics
print("\n" + "=" * 50)
print("ANALYSIS COMPLETE")
print("=" * 50)
print(f"Figures saved: {len(list(CONFIG.fig_dir.glob('*.png')))}")
print(f"Output directory: {CONFIG.fig_dir.absolute()}")
print()
print("This EDA characterizes data properties for downstream modeling.")
print("No predictions or models were fit - that belongs in a separate notebook.")